We evaluate whether Retrieval-Augmented Generation (RAG) improves the ability of an LLM (Mistral-7B-Instruct-v0.3) to rewrite unsuccessful Kickstarter campaign descriptions into successful ones.
Here we decided to compare two generation settings:
- Baseline (no RAG): the model rewrites an unsuccessful description using only an instruction prompt.
- RAG: the model is provided with a similar successful campaign (retrieved from the dataset) as an additional example in the prompt.

The objective is to measure whether including a retrieved successful example leads to generated descriptions that are more likely to be classified as successful.
The newly generated descriptions are evaluated using the previously trained classification model (BERT), which assigns a success probability to each output.

We first import the necessary modules


In [1]:
import pandas as pd
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_distances
from transformers import BitsAndBytesConfig, AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We load the processed Kickstarter dataset and compute the word count of each campaign description.
To keep LLM prompts manageable and reduce computational cost, we restrict the candidate pool to unsuccessful campaigns (status = 0) with fewer than 500 words.
From this filtered subset, we then randomly sample 5 descriptions to be rewritten by the generative model. The fixed random seed ensures that the sample is reproducible.

In [5]:
data = pd.read_csv('Kickstarter_processed.csv')
data["word_count"] = data["description"].fillna("").apply(lambda x: len(x.split()))

sample = data[data['word_count'] < 500]
sample = sample[sample['status'] == 0]



sample = sample.groupby('category', group_keys=False).sample(
    n=20, replace=False, random_state=67
)
sample[['category', 'description', 'word_count']]

,category,description,word_count
4733,Film & Video,This story is the 1st act in the feature film ...,248
4745,Film & Video,Welcome to the Bakersfield Wink Film Festival ...,144
6112,Film & Video,Hey ya'll! I'm Elisa but my friends call me El...,376
2997,Film & Video,"The Black Madonna is a short, experimental fil...",364
1862,Film & Video,"Why do we need the ""Joy Circles Gang? It's in ...",304
...,...,...,...
2250,Technology,The vision and goals of Agernation We are deve...,293
316,Technology,Phones. They're a part of our everyday lives. ...,447
37,Technology,Introducing the Vividia G-810N BlueEye Grabber...,406
3613,Technology,The Open Media Foundation's community space in...,174


In [6]:
sample['category'].value_counts()


category
Film & Video    20
Games           20
Music           20
Publishing      20
Technology      20
Name: count, dtype: int64

Next, we use all-MiniLM-L6-v2, a SentenceTransformer model, to represent each campaign description as a dense embedding vector.
For the RAG setting, we build a retrieval index containing only successful campaigns (status = 1). The function get_most_similar_successful_example() takes an unsuccessful campaign as input and retrieves the most semantically similar successful campaign from the same category.

Similarity is computed using cosine distance between normalized sentence embeddings. To avoid retrieving examples that are too different in length, we restrict the candidate successful campaigns to descriptions between 0.5 and 1.5 times the length of the target description.

The retrieved successful campaign is later inserted into the prompt as an example to guide the model during the rewriting process.

In [7]:
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2" # load the embedding model 
embed_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embed_model = SentenceTransformer(EMBED_MODEL_ID, device=embed_device)

def encode_texts(texts, batch_size=32, max_length=512): #turns list of text into embedding vectors
    text_list = ["" if pd.isna(text) else str(text) for text in texts]

    embed_model.max_seq_length = max_length

    embeddings = embed_model.encode(
        text_list,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True)
    
    return embeddings.astype('float32')

search_source = data.copy()
if "word_count" not in search_source.columns:
    search_source["word_count"] = search_source["description"].fillna("").astype(str).str.split().str.len()

successful = search_source[search_source["status"] == 1].copy()
successful_example_index = {}

for category_name, category_df in successful.groupby("category"):
    category_df = category_df.reset_index(drop=False).rename(columns={"index": "source_index"}).reset_index(drop=True)
    successful_example_index[category_name] = {
        "frame": category_df,
        "embeddings": encode_texts(category_df["description"].fillna("").astype(str).tolist()),
    }


def get_most_similar_successful_example(row, min_len_ratio=0.5, max_len_ratio=1.5, top_k=1):
    category_name = row["category"]
    if category_name not in successful_example_index:
        return pd.DataFrame()

    pool = successful_example_index[category_name]["frame"].copy()
    pool_embeddings = successful_example_index[category_name]["embeddings"]

    target_text = "" if pd.isna(row["description"]) else str(row["description"])
    target_length = len(target_text.split())

    length_mask = pool["word_count"].between(
        target_length * min_len_ratio,
        target_length * max_len_ratio,
    )

    if length_mask.any():
        pool = pool.loc[length_mask].reset_index(drop=True)
        pool_embeddings = pool_embeddings[length_mask.to_numpy()]

    target_embedding = encode_texts([target_text])
    distances = cosine_distances(target_embedding, pool_embeddings)[0]

    ranked = pool.assign(
        cosine_distance=distances,
        cosine_similarity=1 - distances,
    ).sort_values("cosine_distance", ascending=True)

    return ranked.head(top_k)


Batches: 100%|██████████| 21/21 [00:03<00:00,  6.38it/s]


We now load the generative model used for rewriting: Mistral-7B-Instruct-v0.3. This is an instruction-tuned model, suitable for our task where the prompt explicitly asks the model to transform an input description.
To reduce memory requirements, we load the model in 4-bit quantized format. 
We set the model generation parameters are set as follows:
- max_new_tokens = 500: limits the maximum length of the generated rewrite.
- temperature = 0.8: introduces moderate randomness. We use a slightly higher value (default is set to 0.7) to reduce the risk of copying too closely from the retrieved example.
- do_sample = True: enables sampling, which is required for temperature to affect generation.
- return_full_text = False: returns only the generated rewrite, not the full input prompt.

In [8]:
# here we load the model but with the parameters quantized 
model_id  = 'mistralai/Mistral-7B-Instruct-v0.3'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=500, #limits the words (we need to put it also in the prompt I think)
        temperature=0.8, # we can see if different temperatures give different results also
        do_sample=True, #it says like its required for the temperature setting to work I guess
        repetition_penalty=1.0, #the higher = more penalization in repetitive output 
        return_full_text=False,
    ),
    model_kwargs={"quantization_config": quantization_config},
)

W0509 22:41:05.474000 5704 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|          | 2/291 [00:00<01:26,  3.33it/s]c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:19<00:00, 15.27it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Below we define a function that transforms chat-style messages into the chat template expected by the model

ROLE_MAP maps message roles (human, ai, system) to the corresponding chat roles (user, assistant, system). The function then applies the model's tokenizer chat template, to ensure that prompts are formatted consistently with the instruction-tuning format of the model.

In [9]:
ROLE_MAP = {"human": "user", "ai": "assistant", "system": "system"}

def chatpromptvalue_to_mistral(pv):
    msgs = pv.to_messages()
    chat = [{"role": ROLE_MAP[m.type], "content": m.content} for m in msgs]
    return llm.pipeline.tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
    )

to_mistral = RunnableLambda(chatpromptvalue_to_mistral) 


We define the baseline prompt (no RAG), where the model is asked to rewrite an unsuccessful campaign description into a more effective one.
The prompt still constrains the model to:
- preserve the original idea of the project
- avoid introducing new factual information
- only improve how the idea is communicated

This ensures that any improvement in the generated text comes from better wording rather than changes in content or a whole different idea being introduced in the project.

In [ ]:
user_prompt = '''Rewrite the unsuccessful Kickstarter campaign in the category {category} to turn it into a successful one.
You can only change how the idea is communicated. 
Do not invent any factual details.
Make sure to do not make any modification to the core idea of the project and preserve the original idea. 
Unsuccessful campaign:
{sentence}

Rewritten version: ''' 

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", '''You are an expert Kickstarter copywriter. Your goal is to rewrite 
         campaign descriptions so they read like likely successful projects'''),
        ("human", user_prompt),
    ]
)

We create the baseline generation chain by combining the prompt, chat template conversion, the LLM, and an output parser.
For each sampled unsuccessful campaign, the chain generates a rewritten version using only the baseline prompt. Since the model may include residual tokens from the chat template (e.g. [INST], </s>) or repeat part of the prompt, we apply a function to remove these elements and retain only the rewritten description. 
The generated outputs are then stored in a dataframe together with the original description and campaign category.

In [11]:
def clean_response(text):
    text = text.strip()

    if "[/INST]" in text:
        text = text.split("[/INST]", 1)[-1].strip()

    if text.startswith("<s>"):
        text = text[3:].strip()

    if text.startswith("[INST]"):
        text = text[len("[INST]"):].strip()

    if "New description:" in text:
        text = text.split("New description:", 1)[-1].strip()

    if text.startswith("Rewritten version:"):
        text = text[len("Rewritten version:"):].strip()

    return text

In [12]:
chain = prompt | to_mistral | llm | StrOutputParser()

generated_rows = []

for i in range(len(sample)):
    row = sample.iloc[i]
    example_df = get_most_similar_successful_example(row, top_k=1)
    example_text = example_df.iloc[0]["description"] if not example_df.empty else ""

    response = clean_response(chain.invoke({
        "sentence": row["description"],
        "category": row["category"],
        "example": example_text,
    }))
    generated_rows.append({
        "sentence": row["description"],
        "category": row["category"],
        "example": example_text,
        "response": response,
    })

generated = pd.DataFrame(generated_rows)
generated

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.88it/s]
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Batches: 100%|██████████| 1/1 [00:00<00:00, 92.82it/s]
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bits

,sentence,category,example,response
0,This story is the 1st act in the feature film ...,Film & Video,WE'VE REACHED OUR FIRST GOAL! Thank you so muc...,"Title: ""BREAKING BARRIERS: A Sci-Fi Saga - Bri..."
1,Welcome to the Bakersfield Wink Film Festival ...,Film & Video,Thanks for your support! Texas Boys is an upco...,Greetings! We're thrilled to introduce the Bak...
2,Hey ya'll! I'm Elisa but my friends call me El...,Film & Video,"THE METAMORPHOSES is a drama about Francis, a ...","Title: ""Your Story, Your Choices: A Journey Th..."
3,"The Black Madonna is a short, experimental fil...",Film & Video,"THE METAMORPHOSES is a drama about Francis, a ...","Title: ""The Black Madonna: A Celebration of Bl..."
4,"Why do we need the ""Joy Circles Gang? It's in ...",Film & Video,Welcome to my biggest dream . I honestly feel ...,"Title: ""Empowering the Preteens: Introducing t..."
...,...,...,...,...
95,The vision and goals of Agernation We are deve...,Technology,"So, what exactly is MedVenture? MedVenture was...",Title: Empowering Lifestyle Enhancement for th...
96,Phones. They're a part of our everyday lives. ...,Technology,Kitables is on a mission to create a world of ...,Titled: Unleash Your Inner Engineer: Build You...
97,Introducing the Vividia G-810N BlueEye Grabber...,Technology,"In our everyday lives, we rarely have the prop...",Introducing the Vividia G-810N BlueEye Grabber...
98,The Open Media Foundation's community space in...,Technology,NanoBeam is a miniature construction set for b...,Title: Revolutionize Your Creativity: Join the...


In [13]:
generated

,sentence,category,example,response
0,This story is the 1st act in the feature film ...,Film & Video,WE'VE REACHED OUR FIRST GOAL! Thank you so muc...,"Title: ""BREAKING BARRIERS: A Sci-Fi Saga - Bri..."
1,Welcome to the Bakersfield Wink Film Festival ...,Film & Video,Thanks for your support! Texas Boys is an upco...,Greetings! We're thrilled to introduce the Bak...
2,Hey ya'll! I'm Elisa but my friends call me El...,Film & Video,"THE METAMORPHOSES is a drama about Francis, a ...","Title: ""Your Story, Your Choices: A Journey Th..."
3,"The Black Madonna is a short, experimental fil...",Film & Video,"THE METAMORPHOSES is a drama about Francis, a ...","Title: ""The Black Madonna: A Celebration of Bl..."
4,"Why do we need the ""Joy Circles Gang? It's in ...",Film & Video,Welcome to my biggest dream . I honestly feel ...,"Title: ""Empowering the Preteens: Introducing t..."
...,...,...,...,...
95,The vision and goals of Agernation We are deve...,Technology,"So, what exactly is MedVenture? MedVenture was...",Title: Empowering Lifestyle Enhancement for th...
96,Phones. They're a part of our everyday lives. ...,Technology,Kitables is on a mission to create a world of ...,Titled: Unleash Your Inner Engineer: Build You...
97,Introducing the Vividia G-810N BlueEye Grabber...,Technology,"In our everyday lives, we rarely have the prop...",Introducing the Vividia G-810N BlueEye Grabber...
98,The Open Media Foundation's community space in...,Technology,NanoBeam is a miniature construction set for b...,Title: Revolutionize Your Creativity: Join the...


In [14]:
generated.to_csv('Kickstarter_new.csv')

We now load the fine-tuned BERT classifier previously finetuned for the campaign success classification task.
The saved tokenizer and model weights are loaded from saved_bert_model. We then set the model to evaluation mode to ensure consistent predictions during inference.
This classifier will be used to estimate the probability that each generated description is classified as successful.

In [15]:
classifier_dir = "final_roberta_fine-tuned"
classifier_config = AutoConfig.from_pretrained(
    classifier_dir,
    id2label={0: "Not successful", 1: "Successful"},
    label2id={"Not successful": 0, "Successful": 1},
)
classifier_tokenizer = AutoTokenizer.from_pretrained(classifier_dir)
classifier_model = AutoModelForSequenceClassification.from_pretrained(
    classifier_dir,
    config=classifier_config,
)
classifier_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier_model.to(classifier_device)
classifier_model.eval()

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3688.69it/s]


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, b

In [16]:
generated = pd.read_csv('Kickstarter_new.csv')
generated 

,Unnamed: 0,sentence,category,example,response
0,0,This story is the 1st act in the feature film ...,Film & Video,WE'VE REACHED OUR FIRST GOAL! Thank you so muc...,"Title: ""BREAKING BARRIERS: A Sci-Fi Saga - Bri..."
1,1,Welcome to the Bakersfield Wink Film Festival ...,Film & Video,Thanks for your support! Texas Boys is an upco...,Greetings! We're thrilled to introduce the Bak...
2,2,Hey ya'll! I'm Elisa but my friends call me El...,Film & Video,"THE METAMORPHOSES is a drama about Francis, a ...","Title: ""Your Story, Your Choices: A Journey Th..."
3,3,"The Black Madonna is a short, experimental fil...",Film & Video,"THE METAMORPHOSES is a drama about Francis, a ...","Title: ""The Black Madonna: A Celebration of Bl..."
4,4,"Why do we need the ""Joy Circles Gang? It's in ...",Film & Video,Welcome to my biggest dream . I honestly feel ...,"Title: ""Empowering the Preteens: Introducing t..."
...,...,...,...,...,...
95,95,The vision and goals of Agernation We are deve...,Technology,"So, what exactly is MedVenture? MedVenture was...",Title: Empowering Lifestyle Enhancement for th...
96,96,Phones. They're a part of our everyday lives. ...,Technology,Kitables is on a mission to create a world of ...,Titled: Unleash Your Inner Engineer: Build You...
97,97,Introducing the Vividia G-810N BlueEye Grabber...,Technology,"In our everyday lives, we rarely have the prop...",Introducing the Vividia G-810N BlueEye Grabber...
98,98,The Open Media Foundation's community space in...,Technology,NanoBeam is a miniature construction set for b...,Title: Revolutionize Your Creativity: Join the...


We now define a function to classify the generated descriptions using the fine-tuned BERT classifier.
The function tokenizes the generated texts in batches, applies truncation at 512 tokens, and obtains the classifier logits without updating model parameters. The logits are converted into probabilities using softmax.

For each generated description, we store the predicted label, the predicted outcome, and the probability assigned to the successful class. This success probability is used as the evaluation score for the generated outputs.

In [17]:
def classify_responses(df, text_col="response", batch_size=16, max_length=512):
    texts = df[text_col].fillna("").astype(str).tolist()
    predicted_labels = []
    success_probabilities = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = classifier_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {key: value.to(classifier_device) for key, value in inputs.items()}

        with torch.no_grad():
            logits = classifier_model(**inputs).logits
            probabilities = torch.softmax(logits, dim=-1)

        predicted_labels.extend(probabilities.argmax(dim=-1).cpu().tolist())
        success_probabilities.extend(probabilities[:, 1].cpu().tolist())

    scored = df.copy()
    scored["predicted_label"] = predicted_labels
    scored["predicted_outcome"] = scored["predicted_label"].map({0: "Not successful", 1: "Successful"})
    scored["success_probability"] = success_probabilities
    return scored

generated = classify_responses(generated, text_col="response")
generated[["category", "response", "predicted_outcome", "success_probability"]]

,category,response,predicted_outcome,success_probability
0,Film & Video,"Title: ""BREAKING BARRIERS: A Sci-Fi Saga - Bri...",Successful,0.880795
1,Film & Video,Greetings! We're thrilled to introduce the Bak...,Successful,0.768133
2,Film & Video,"Title: ""Your Story, Your Choices: A Journey Th...",Successful,0.674725
3,Film & Video,"Title: ""The Black Madonna: A Celebration of Bl...",Successful,0.967026
4,Film & Video,"Title: ""Empowering the Preteens: Introducing t...",Not successful,0.390180
...,...,...,...,...
95,Technology,Title: Empowering Lifestyle Enhancement for th...,Not successful,0.298731
96,Technology,Titled: Unleash Your Inner Engineer: Build You...,Successful,0.715507
97,Technology,Introducing the Vividia G-810N BlueEye Grabber...,Successful,0.726372
98,Technology,Title: Revolutionize Your Creativity: Join the...,Successful,0.687378


In [18]:
generated["predicted_outcome"].value_counts()

predicted_outcome
Successful        53
Not successful    47
Name: count, dtype: int64

We define the RAG prompt, where the model is provided with a similar successful campaign as an additional input.
The retrieved example is used only as stylistic and structural guidance, while the model is constrained to preserve the original idea and avoid copying content or introducing new factual information.
This allows us to test whether conditioning the model on a relevant successful example improves the quality of the generated rewrite.

In [23]:
user_prompt_RAG = '''Rewrite the unsuccessful Kickstarter campaign in the category {category} 
using the successful example only as stylistic and structural guidance.

Do not copy phrases.
Do not invent factual details.
Preserve the original project idea.

Unsuccessful campaign:
{sentence}

You can study the most similar successful campaign as an example and adopt hooks, structure, tone,
signals, call to action and other useful patterns from the example. 
Do not copy the exact wording, do not invent new facts. 
Most similar successful campaign:
{example}

Output only the rewritten campaign
Rewritten version: ''' 

# The full prompt, in chat format. In this case, we are in the first
# conversation turn and therefore we only have the sytem and the user prompts.
prompt_RAG = ChatPromptTemplate.from_messages(
    [
        ("system", '''You are an expert Kickstarter copywriter. Your goal is to rewrite 
         campaign descriptions so they read like likely successful projects. You only need to 
         output the rewritten campaign'''),
        ("human", user_prompt_RAG),
    ]
)

We create the RAG generation chain using the RAG prompt, just as we did before in the non-RAG case. 
For each sampled unsuccessful campaign, we retrieve the most semantically similar successful campaign from the same category. The retrieved campaign is inserted into the prompt as an example, and the model generates a rewritten version of the unsuccessful campaign.

The output is cleaned to extract only the rewritten description, and the final results are stored together with the original campaign, category, retrieved example, and generated response.

In [ ]:
chain_RAG = prompt_RAG | to_mistral | llm | StrOutputParser()

generated_rows_RAG = []

for i in range(len(sample)):
    row = sample.iloc[i]
    example_df_rag = get_most_similar_successful_example(row, top_k=1)
    example_text_rag = example_df_rag.iloc[0]["description"] if not example_df_rag.empty else ""

    response_rag = clean_response(chain_RAG.invoke({
        "sentence": row["description"],
        "category": row["category"],
        "example": example_text_rag,
    }))
    generated_rows_RAG.append({
        "sentence": row["description"],
        "category": row["category"],
        "example": example_text_rag,
        "response": response_rag,
    })

generated_rag = pd.DataFrame(generated_rows_RAG)
generated_rag

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.99it/s]
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Batches: 100%|██████████| 1/1 [00:00<00:00, 140.32it/s]
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bit

,sentence,category,example,response
0,This story is the 1st act in the feature film ...,Film & Video,WE'VE REACHED OUR FIRST GOAL! Thank you so muc...,Title: Unraveling the Enigma: A captivating Sc...
1,Welcome to the Bakersfield Wink Film Festival ...,Film & Video,Thanks for your support! Texas Boys is an upco...,Welcome to the Bakersfield Visionaries Film Fe...
2,Hey ya'll! I'm Elisa but my friends call me El...,Film & Video,"THE METAMORPHOSES is a drama about Francis, a ...",Title: Shifting Realities: An Immersive Experi...
3,"The Black Madonna is a short, experimental fil...",Film & Video,"THE METAMORPHOSES is a drama about Francis, a ...","Titled ""The Divine Reclamation: A Luminous Tal..."
4,"Why do we need the ""Joy Circles Gang? It's in ...",Film & Video,Welcome to my biggest dream . I honestly feel ...,"Embrace the Adventure with ""Adira's Tales"": A ..."


We apply the BERT classifier to the RAG-generated descriptions to estimate their likelihood of being successful.

Each generated response is evaluated by the classifier, which assigns a predicted label and a probability of success. These probabilities are used to compare the effectiveness of the RAG-based generation against the baseline approach.

In [33]:
generated_rag= classify_responses(generated_rag, text_col="response")
generated_rag[["category", "response", "predicted_outcome", "success_probability"]]

,category,response,predicted_outcome,success_probability
0,Film & Video,Title: Unraveling the Enigma: A captivating Sc...,Successful,0.804882
1,Film & Video,Welcome to the Bakersfield Visionaries Film Fe...,Successful,0.799271
2,Film & Video,Title: Shifting Realities: An Immersive Experi...,Not successful,0.439796
3,Film & Video,"Titled ""The Divine Reclamation: A Luminous Tal...",Successful,0.900132
4,Film & Video,"Embrace the Adventure with ""Adira's Tales"": A ...",Successful,0.883595


In [34]:
generated_rag["predicted_outcome"].value_counts()

predicted_outcome
Successful        4
Not successful    1
Name: count, dtype: int64

In [ ]:
comparison = pd.DataFrame({
    "category": generated["category"],
    "original_description":generated["sentence"],
    "baseline_response": generated["response"],
    "baseline_success_probability":generated["success_probability"],
    "rag_response": generated_rag["response"],
    "rag_success_probability": generated_rag["success_probability"],
    "retrieved_example": generated_rag["example"],
})

comparison["rag_minus_baseline"] = (
    comparison["rag_success_probability"] 
    - comparison["baseline_success_probability"]
)

comparison["better_method"] = comparison["rag_minus_baseline"].apply(
    lambda x: "RAG" if x > 0 else "Baseline"
)

comparison[[
    "category",
    "baseline_success_probability",
    "rag_success_probability",
    "rag_minus_baseline",
    "better_method"
]]

In [ ]:
summary = pd.DataFrame({
    "method": ["Baseline", "RAG"],
    "mean_success_probability": [
        comparison["baseline_success_probability"].mean(),
        comparison["rag_success_probability"].mean()
    ],
    "median_success_probability": [
        comparison["baseline_success_probability"].median(),
        comparison["rag_success_probability"].median()
    ],
    "num_predicted_successful": [
        (generated["predicted_label"] == 1).sum(),
        (generated_rag["predicted_label"] == 1).sum()
    ]
})

summary